# FLAG 2027 — NB-13: external-data EDA + curation (deep-research-report 29/09, reduced plan)

Three parts, all on frozen features (no backbone runs), TRUE labels:

| Part | Question | Output |
|---|---|---|
| **EDA-EXT-01** | How much *independent* information do MAV-Celeb v1/v2 persons really add (unique clips, videos, languages)? | `eda_ext_persons.csv`, `eda_ext_summary.json` |
| **EDA-SHIFT-01** | Where do v1/v2 differ from v4 (face / voice / attributes), and how separable is language inside the same persons? Classifier two-sample test, **split by person** | `shift_c2st.csv` |
| **CUR-01** | Does a *curated* external set beat an equal-size random one, and does language selection matter? Production member (s007 + ImageBind + age) scored like SET-13 | `cur_results.csv`, `cur_summary.csv` |

**CUR-01 arms** (registered before running; 70 v4 persons + v1/v2 persons of the 4 folds not held out; every arm the SAME
number of optimizer steps, P x K = 32 x 8, EXP-007 head, 3 nets, seeds fold + 100 i; the folds are those of EXTSCALE-01):

| Arm | External rows used for training |
|---|---|
| `R` | none (v4 only) |
| `XALL` | all languages (<= 150 rows per person) |
| `XEN` | English rows only |
| `XEN_MATCH` | English rows of the 50 % of persons most like v4 train (out-of-fold domain classifier on person means of VGG + BTC) |
| `XEN_RAND` | English rows of a random 50 % of persons (same count as `XEN_MATCH`) |
| `XLANG_EQ` | all persons, English and Urdu/Hindi rows balanced per person (<= 75 each) |

Pre-registered comparisons: `XEN_MATCH` vs `XEN_RAND` (does curation help?), `XEN` vs `XALL` (does the native language
help or hurt?), `XLANG_EQ` vs `XALL` (does balance matter?), each vs `R`. Decision metric: **production, person and cluster
level**; gate >= 1 EER in >= 4/5 folds (DECISIONS A3 / A7).

**Settings:** GPU T4 (x1 used) or CPU; Internet OFF is fine (no downloads). **Input:** the dataset `flag2027-eda-bundle`.
**Time:** GPU T4 about 30-45 min; CPU (4 cores) about 2-3 h. **Output:** `/kaggle/working/eda_cur/` (download it as a zip).
**Resume:** CUR-01 appends to `cur_results.csv` after each fold x arm; attach this notebook's earlier output to skip
finished pairs.

In [ ]:
import os, sys, re, io, json, time, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
LOCAL = os.environ.get("FLAG_LOCAL") == "1"
IN = Path(os.environ.get("FLAG_IN", "/kaggle/input"))
OUT = Path(os.environ.get("FLAG_OUT", "/kaggle/working")) / "eda_cur"
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(os.cpu_count() or 4)

# ------------------------------------------------------------------ registered settings (do not tune after looking)
STEPS = int(os.environ.get("CUR_STEPS", "1200"))      # optimizer steps of EVERY arm
N_NETS = int(os.environ.get("CUR_NETS", "3"))
FOLDS = [int(x) for x in os.environ.get("CUR_FOLDS", "1,2,3,4,5").split(",")]
ARMS = ["R", "XALL", "XEN", "XEN_MATCH", "XEN_RAND", "XLANG_EQ"]
P_PK, K_PK, CAP, CAP_LANG = 32, 8, 150, 75
TF_CLUS, TV_CLUS = 0.75, 0.70                          # production cluster thresholds (v4 train calibration)
W_IB, W_AGE, W_CCA = 0.5, 0.1, 0.25                    # SET-13
OVERLAP = {"v1_complete:shaukat_tarin", "v2_complete:id0073"}   # persons also in v4 (SET-07)
AGE_MIDS = np.array([1, 6, 15, 25, 35, 45, 55, 65, 75], dtype=np.float32)
print("LOCAL" if LOCAL else "KAGGLE", "| device", DEVICE, "| steps", STEPS, "| nets", N_NETS, "| folds", FOLDS)


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

## 1. Data (one bundle)

In [ ]:
# ============================================================================ data (one bundle)
man = next(IN.rglob("eda_bundle_manifest.json"), None)
assert man is not None, "attach the dataset flag2027-eda-bundle (kaggle/build_eda_bundle.py)"
B = man.parent
A = lambda n: np.load(B / f"{n}.npy")
T = lambda n: pd.read_csv(B / n)
log("bundle:", B, "|", json.loads(man.read_text())["created"])

# v4 train: one row = the face and the voice of one clip
V4 = dict(vgg=A("v4_face_vgg"), btc=A("v4_voice_btc"), ibf=A("v4_face_ibv").astype(np.float32),
          iba=A("v4_voice_iba").astype(np.float32), agef=A("v4_face_vitageprob") @ AGE_MIDS,
          agev=A("v4_voice_w2v2agage")[:, 0] * 100, farl=A("v4_face_farl").astype(np.float32),
          arc=A("v4_face_arcface").astype(np.float32), ecapa=A("v4_voice_ecapa192").astype(np.float32))
V4S = T("v4_rows.csv").spk.astype(str).values
GMAP = dict(zip(T("v4_gender.csv").spk, T("v4_gender.csv").gender))

# v1 / v2: faces and voices in separate tables; persons merged by name (v1 lists some people twice)
FM_, VM_, FA_, VA_, GEN = [], [], {}, {}, {}
for src in ("v1_complete", "v2_complete"):
    fm, vm = T(f"ext_{src}_face_meta.csv"), T(f"ext_{src}_voice_meta.csv")
    key_of = {}
    sp = B / f"ext_{src}_speakers.csv"
    if sp.exists():
        for r in pd.read_csv(sp).itertuples():
            k = f"{src}:" + str(r.name).strip().lower().replace(" ", "_")
            key_of[r.ids] = k
            GEN[k] = "m" if str(r.gender).lower().startswith("m") else "f"
    key = lambda x, key_of=key_of, src=src: key_of.get(x, f"{src}:{x}")
    FM_.append(fm.assign(p=fm.spk.map(key), src=src)); VM_.append(vm.assign(p=vm.spk.map(key), src=src))
    for k in ("vgg", "ibv", "vitageprob", "farl", "arcface"):
        FA_.setdefault(k, []).append(A(f"ext_{src}_face_{k}").astype(np.float32))
    for k in ("btc", "iba", "w2v2agage", "w2v2aggen", "ecapa192"):
        VA_.setdefault(k, []).append(A(f"ext_{src}_voice_{k}").astype(np.float32))
FMETA, VMETA = pd.concat(FM_, ignore_index=True), pd.concat(VM_, ignore_index=True)
FA = {k: np.concatenate(v) for k, v in FA_.items()}; VA = {k: np.concatenate(v) for k, v in VA_.items()}
FA["agef"] = FA["vitageprob"] @ AGE_MIDS; VA["agev"] = VA["w2v2agage"][:, 0] * 100
male = VA["w2v2aggen"][:, 1] > VA["w2v2aggen"][:, 0]                 # columns (female, male, child)
for p, g in VMETA.assign(m=male).groupby("p").m.mean().items():
    GEN.setdefault(p, "m" if g >= .5 else "f")                       # v2: majority of the predicted calls
GMAP.update(GEN)
PERSONS = sorted(set(VMETA.p) - OVERLAP)
PFOLD = dict(zip(PERSONS, np.random.RandomState(0).permutation(len(PERSONS)) % 5))   # = EXTSCALE-01 folds
log(f"v4 {len(V4S)} rows / {len(set(V4S))} persons | v1+v2 faces {len(FMETA)}, voices {len(VMETA)}, persons {len(PERSONS)}")

## 2. Bridge and scoring code (port of kaggle/flag_v2.py; bit-identical to it, checked locally)

In [ ]:
# ============================================================================ bridge + scoring (port of kaggle/flag_v2.py, EXP-007 recipe)
REC = dict(pca_f=256, pca_v=None, hid=512, emb=128, drop=0.3, lr=1e-3, wd=1e-2, tau=0.07, epochs=40)
l2n = lambda X: X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
zm = lambda M: (M - M.mean()) / (M.std() + 1e-9)


class Prep:
    """Standardise; PCA the face to 256 (fitted on training rows)."""
    def __init__(self, Xf, Xv, pca_f=256, pca_v=None):
        self.mf, self.sf = Xf.mean(0), Xf.std(0) + 1e-6
        self.mv, self.sv = Xv.mean(0), Xv.std(0) + 1e-6
        self.Pf = self._pca((Xf - self.mf) / self.sf, pca_f)
        self.Pv = self._pca((Xv - self.mv) / self.sv, pca_v)

    @staticmethod
    def _pca(Z, k):
        if not k or k >= Z.shape[1]:
            return None
        _, _, Vt = np.linalg.svd(Z.astype(np.float64), full_matrices=False)
        return Vt[:k].T.astype(np.float32)

    def f(self, X):
        Z = (X - self.mf) / self.sf
        return (Z if self.Pf is None else Z @ self.Pf).astype(np.float32)

    def v(self, X):
        Z = (X - self.mv) / self.sv
        return (Z if self.Pv is None else Z @ self.Pv).astype(np.float32)


def branch(d_in, emb, drop, hid):
    return nn.Sequential(nn.Dropout(drop), nn.Linear(d_in, hid), nn.BatchNorm1d(hid), nn.ReLU(), nn.Dropout(drop), nn.Linear(hid, emb))


class Model(nn.Module):
    def __init__(self, d_f, d_v, n_cls, cfg):
        super().__init__()
        self.bf = branch(d_f, cfg["emb"], cfg["drop"], cfg["hid"])
        self.bv = branch(d_v, cfg["emb"], cfg["drop"], cfg["hid"])
        self.W = nn.Parameter(torch.randn(n_cls, cfg["emb"]) * 0.01)    # unused here; kept so the RNG stream matches flag_v2
        self.gdisc = nn.Sequential(nn.Linear(cfg["emb"], 64), nn.ReLU(), nn.Linear(64, 2))
        self.ldisc = nn.Sequential(nn.Linear(cfg["emb"], 64), nn.ReLU(), nn.Linear(64, 2))

    def forward(self, f, v):
        return F.normalize(self.bf(f), dim=1), F.normalize(self.bv(v), dim=1)


def infonce(u, w, y, tau):
    S = u @ w.T / tau
    same = (y[:, None] == y[None, :]).float()
    return 0.5 * ((-(F.log_softmax(S, 1) * same).sum(1) / same.sum(1)).mean() +
                  (-(F.log_softmax(S, 0) * same).sum(0) / same.sum(0)).mean())


def pk_batches(spk_idx, P, K, n_rows, rng):
    speakers = list(spk_idx)
    out = []
    for _ in range(max(1, n_rows // (P * K))):
        chosen = rng.choice(len(speakers), size=min(P, len(speakers)), replace=False)
        out.append(np.concatenate([rng.choice(spk_idx[speakers[c]], size=K, replace=len(spk_idx[speakers[c]]) < K)
                                   for c in chosen]))
    return out


def train_one(TF, TV, spk, seed, epochs, P=P_PK, K=K_PK):
    """= flag_v2.train_one(cfg = EXP-007 recipe + sampler='pk', rows=(TF, TV, spk)); InfoNCE only."""
    cfg = dict(REC, epochs=epochs)
    torch.manual_seed(seed); np.random.seed(seed)
    rng = np.random.RandomState(seed)
    prep = Prep(TF, TV, cfg["pca_f"], cfg["pca_v"])
    Ftr = torch.tensor(prep.f(TF), device=DEVICE)
    Vv = torch.stack([torch.tensor(prep.v(TV), device=DEVICE)])
    cls = {s: i for i, s in enumerate(np.unique(spk))}
    Y = torch.tensor([cls[s] for s in spk], device=DEVICE)
    net = Model(Ftr.shape[1], Vv.shape[2], len(cls), cfg).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, cfg["epochs"])
    n = len(spk)
    spk_idx = {s: np.where(np.asarray(spk) == s)[0] for s in np.unique(spk)}
    P = min(P, len(spk_idx))
    for ep in range(cfg["epochs"]):
        net.train()
        rng.permutation(n); vsel = np.zeros(n, dtype=int)                 # same RNG calls as flag_v2 (full view)
        for b in pk_batches(spk_idx, P, K, n, rng):
            bt = torch.tensor(b, device=DEVICE); vt = torch.tensor(vsel[b], device=DEVICE)
            u, w = net(Ftr[bt], Vv[vt, bt])
            loss = infonce(u, w, Y[bt], cfg["tau"])
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    net.eval()
    return net, prep


@torch.no_grad()
def embed(net, prep, Af, Av, mu_f, mu_v):
    """Per-file centring (file mean -> train mean), then the two branches (as flag_v2.embed)."""
    A_ = prep.f(Af); B_ = prep.v(Av)
    A_ = torch.tensor(A_ - A_.mean(0) + mu_f, device=DEVICE); B_ = torch.tensor(B_ - B_.mean(0) + mu_v, device=DEVICE)
    u, w = net(A_, B_)
    return u.cpu().numpy(), w.cpu().numpy()


class RidgeCCA:
    def __init__(self, k=4, reg=1.0, pca_x=128, pca_y=192):
        self.k, self.reg, self.pca_x, self.pca_y = k, reg, pca_x, pca_y

    def _prep(self, X, n_pca):
        mu, sd = X.mean(0), X.std(0) + 1e-6
        Z = (X - mu) / sd
        if n_pca is not None and n_pca < Z.shape[1]:
            _, _, Vt = np.linalg.svd(Z, full_matrices=False); P = Vt[:n_pca].T
        else:
            P = np.eye(Z.shape[1])
        return mu, sd, P

    def fit(self, X, Y):
        self.mx, self.sx, self.Px = self._prep(X, min(self.pca_x, X.shape[1]) if self.pca_x else None)
        self.my, self.sy, self.Py = self._prep(Y, min(self.pca_y, Y.shape[1]) if self.pca_y else None)
        A_ = ((X - self.mx) / self.sx) @ self.Px; B_ = ((Y - self.my) / self.sy) @ self.Py
        n = len(A_)
        Cxx = A_.T @ A_ / n + self.reg * np.eye(A_.shape[1]); Cyy = B_.T @ B_ / n + self.reg * np.eye(B_.shape[1])
        Lxi = np.linalg.inv(np.linalg.cholesky(Cxx)); Lyi = np.linalg.inv(np.linalg.cholesky(Cyy))
        U, S, Vt = np.linalg.svd(Lxi @ (A_.T @ B_ / n) @ Lyi.T, full_matrices=False)
        self.Wx = Lxi.T @ U[:, :self.k]; self.Wy = Lyi.T @ Vt[:self.k].T
        return self

    def transform_x(self, X): return l2n((((X - self.mx) / self.sx) @ self.Px) @ self.Wx)
    def transform_y(self, Y): return l2n((((Y - self.my) / self.sy) @ self.Py) @ self.Wy)


def eer(scores, labels):
    """EER %, higher = same; the FAR = FRR crossing is interpolated (as flag_lib.eer_from_scores)."""
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr; d = fnr - fpr
    i = np.nanargmin(np.abs(d))
    if d[i] == 0 or len(fpr) < 2:
        return 100 * (fpr[i] + fnr[i]) / 2
    j = i - 1 if (i > 0 and d[i - 1] * d[i] < 0) else min(i + 1, len(d) - 1)
    if d[j] * d[i] >= 0:
        return 100 * (fpr[i] + fnr[i]) / 2
    t = d[i] / (d[i] - d[j])
    return 100 * float(fpr[i] + t * (fpr[j] - fpr[i]))


def cent(Z):
    return l2n(l2n(Z) - l2n(Z).mean(0))


def clus(Z, t):
    from sklearn.cluster import AgglomerativeClustering
    return AgglomerativeClustering(None, metric="cosine", linkage="average", distance_threshold=t).fit_predict(Z)


def block(M, cf, cv, b=0.99):
    Af = np.zeros((cf.max() + 1, len(cf))); Af[cf, np.arange(len(cf))] = 1; Af /= Af.sum(1, keepdims=True)
    Av = np.zeros((cv.max() + 1, len(cv))); Av[cv, np.arange(len(cv))] = 1; Av /= Av.sum(1, keepdims=True)
    return (1 - b) * np.diag(M) + b * (Af @ M @ Av.T)[cf, cv]

## 3. EDA-EXT-01: independent information per person

In [ ]:
# ============================================================================ EDA-EXT-01: independent information per person
def uniq_rows(X):
    return len(np.unique(np.round(X.astype(np.float32), 3), axis=0))


rows = []
for p, g in VMETA[VMETA.p.isin(PERSONS)].groupby("p"):
    f = FMETA[FMETA.p == p]
    vc = g.video.value_counts(normalize=True).values
    r = dict(person=p, source=g.src.iloc[0], gender=GEN.get(p, "u"), voice_rows=len(g), voice_unique=uniq_rows(VA["btc"][g.index]),
             face_rows=len(f), face_unique=uniq_rows(FA["vgg"][f.index]), videos=g.video.nunique(),
             videos_eff=float(np.exp(-(vc * np.log(vc)).sum())), largest_video_frac=float(vc.max()),
             dur_median=float(g.dur.median()) if "dur" in g else np.nan)
    for lang, n in g.lang.value_counts().items():
        r[f"n_{lang}"] = int(n)
        r[f"videos_{lang}"] = int(g[g.lang == lang].video.nunique())
    rows.append(r)
E = pd.DataFrame(rows).fillna(0)
E["dup_frac"] = 1 - E.voice_unique / E.voice_rows
E.to_csv(OUT / "eda_ext_persons.csv", index=False)
v4u = pd.DataFrame({"spk": V4S}).assign(i=np.arange(len(V4S))).groupby("spk").i.apply(
    lambda ix: (len(ix), uniq_rows(V4["btc"][ix.values]))).apply(pd.Series)
summary = {
    "v4": dict(persons=len(v4u), rows=int(v4u[0].sum()), unique_voice=int(v4u[1].sum()),
               unique_voice_per_person_median=float(v4u[1].median()), unique_voice_per_person_min=int(v4u[1].min())),
    "ext": dict(persons=len(E), by_source=E.source.value_counts().to_dict(), by_gender=E.gender.value_counts().to_dict(),
                voice_rows=int(E.voice_rows.sum()), voice_unique=int(E.voice_unique.sum()),
                videos_median=float(E.videos.median()), videos_eff_median=float(E.videos_eff.median()),
                bilingual_persons=int(((E.get("n_english", 0) > 0) & ((E.get("n_urdu", 0) > 0) | (E.get("n_hindi", 0) > 0))).sum()),
                persons_with_ge2_english_videos=int((E.get("videos_english", 0) >= 2).sum()),
                dup_frac_median=float(E.dup_frac.median()))}
(OUT / "eda_ext_summary.json").write_text(json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1))
print(E.describe().round(2).T[["mean", "50%", "min", "max"]].to_string())

## 4. EDA-SHIFT-01: where v1/v2 differ from v4; how separable language is inside the same persons

In [ ]:
# ============================================================================ EDA-SHIFT-01: classifier two-sample test, split by PERSON
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score


def c2st(X, y, groups, per_group=20, seed=0):
    """AUC of a held-out-person classifier (0.5 = indistinguishable). At most `per_group` rows per person."""
    rng = np.random.RandomState(seed)
    keep = np.concatenate([rng.permutation(np.where(groups == g)[0])[:per_group] for g in np.unique(groups)])
    X, y, groups = X[keep], y[keep], groups[keep]
    X = X.reshape(len(X), -1)
    k = min(64, X.shape[1], len(X) // 5)
    pred = np.zeros(len(y))
    for tr, te in GroupKFold(5).split(X, y, groups):
        clf = make_pipeline(StandardScaler(), *( [PCA(k, random_state=0)] if X.shape[1] > k else []), LogisticRegression(max_iter=2000, C=0.1))
        clf.fit(X[tr], y[tr]); pred[te] = clf.predict_proba(X[te])[:, 1]
    return float(roc_auc_score(y, pred))


ext_ok = VMETA.p.isin(PERSONS).values; fext_ok = FMETA.p.isin(PERSONS).values
en = (VMETA.lang == "english").values
tests = []
face_feats = {"VGG": ("vgg", "vgg"), "ArcFace": ("arc", "arcface"), "FaRL": ("farl", "farl"), "ImageBind face": ("ibf", "ibv"),
              "age (face)": ("agef", "agef")}
voice_feats = {"BTC ECAPA": ("btc", "btc"), "ECAPA-192": ("ecapa", "ecapa192"), "ImageBind audio": ("iba", "iba"),
               "age (voice)": ("agev", "agev")}
for name, (k4, ke) in face_feats.items():                  # v4 vs v1/v2, faces
    X = np.concatenate([V4[k4].reshape(len(V4S), -1), FA[ke][fext_ok].reshape(fext_ok.sum(), -1)])
    y = np.r_[np.zeros(len(V4S)), np.ones(fext_ok.sum())]; g = np.r_[V4S, FMETA.p.values[fext_ok]]
    tests.append(dict(test="v4 vs v1/v2", modality="face", feature=name, auc=c2st(X, y, g)))
for name, (k4, ke) in voice_feats.items():                 # v4 vs v1/v2 English voices; English vs native language (same persons)
    m = ext_ok & en
    X = np.concatenate([V4[k4].reshape(len(V4S), -1), VA[ke][m].reshape(m.sum(), -1)])
    y = np.r_[np.zeros(len(V4S)), np.ones(m.sum())]; g = np.r_[V4S, VMETA.p.values[m]]
    tests.append(dict(test="v4 vs v1/v2 (English voices)", modality="voice", feature=name, auc=c2st(X, y, g)))
    X = VA[ke][ext_ok].reshape(ext_ok.sum(), -1); y = (~en[ext_ok]).astype(int); g = VMETA.p.values[ext_ok]
    tests.append(dict(test="English vs Urdu/Hindi (v1/v2 voices)", modality="voice", feature=name, auc=c2st(X, y, g)))
for name, (k4, ke) in {**face_feats, **voice_feats}.items():   # v1 vs v2
    meta, arr, ok = (FMETA, FA, fext_ok) if name in face_feats else (VMETA, VA, ext_ok)
    X = arr[ke][ok].reshape(ok.sum(), -1); y = (meta.src.values[ok] == "v2_complete").astype(int); g = meta.p.values[ok]
    tests.append(dict(test="v1 vs v2", modality="face" if name in face_feats else "voice", feature=name, auc=c2st(X, y, g)))
C2 = pd.DataFrame(tests); C2.to_csv(OUT / "shift_c2st.csv", index=False)
print(C2.pivot_table(index=["modality", "feature"], columns="test", values="auc").round(3).to_string())

## 5. CUR-01: curated vs random vs all external rows, equal optimizer budget

In [ ]:
# ============================================================================ CUR-01: curated vs random vs all, equal optimizer budget
def ext_rows(persons, langs=None, cap=CAP, per_lang=None, rng=None):
    """Organiser-space training rows (as flag_v2.pair_ext): one per voice clip, paired with a face of the SAME video
    (cycling; any face of the person if that video has none). langs: keep only these languages; per_lang: cap per
    (person, language) instead of per person."""
    vm = VMETA[VMETA.p.isin(persons)]
    if langs is not None:
        vm = vm[vm.lang.isin(langs)]
    fm = FMETA[FMETA.p.isin(persons)]
    by_vid = {k: np.asarray(g.index) for k, g in fm.groupby(["p", "lang", "video"])}
    by_p = {k: np.asarray(g.index) for k, g in fm.groupby("p")}
    Fi, Vi, S = [], [], []
    for key, g in vm.groupby(["p", "lang"] if per_lang else ["p"]):
        p = key[0] if isinstance(key, tuple) else key
        if p not in by_p:
            continue
        idx = np.asarray(g.index); c = per_lang or cap
        if len(idx) > c:
            idx = rng.choice(idx, c, replace=False)
        for j, i in enumerate(idx):
            cand = by_vid.get((p, VMETA.lang[i], VMETA.video[i]), by_p[p])
            Fi.append(cand[j % len(cand)]); Vi.append(i); S.append(p)
    return FA["vgg"][Fi], VA["btc"][Vi], np.array(S)


def v4_likeness(pool, seed=0):
    """Out-of-fold P(v4) for each pool person: classifier on person means of [VGG PCA, BTC], English clips, grouped by person."""
    from sklearn.model_selection import GroupKFold
    vm = VMETA[VMETA.p.isin(pool) & (VMETA.lang == "english")]
    fmean = FMETA[FMETA.p.isin(pool)].groupby("p").apply(lambda g: FA["vgg"][g.index].mean(0))
    vmean = vm.groupby("p").apply(lambda g: VA["btc"][g.index].mean(0))
    ps = [p for p in pool if p in fmean.index and p in vmean.index]
    v4p = np.unique(V4S)
    Xe = np.stack([np.r_[fmean[p], vmean[p]] for p in ps])
    Xv = np.stack([np.r_[V4["vgg"][V4S == s].mean(0), V4["btc"][V4S == s].mean(0)] for s in v4p])
    X = np.concatenate([Xv, Xe]); y = np.r_[np.zeros(len(v4p)), np.ones(len(ps))]; g = np.r_[v4p, ps]
    pred = np.zeros(len(y))
    for tr, te in GroupKFold(5).split(X, y, g):
        clf = make_pipeline(StandardScaler(), PCA(min(32, len(tr) - 1), random_state=seed), LogisticRegression(max_iter=2000, C=0.1))
        clf.fit(X[tr], y[tr]); pred[te] = clf.predict_proba(X[te])[:, 0]
    return pd.Series(pred[len(v4p):], index=ps)                        # P(v4) for the external persons


def eval_file(held, lang, sg, rng, m=16):
    """Held-out persons' voices in `lang` (<= m each); the face of a positive from ANOTHER video (as EXTSCALE-01)."""
    vm = VMETA[(VMETA.lang == lang) & VMETA.p.isin(held)]
    Fi, Vi, Yl, nf, nv = [], [], [], [], []
    for p, g in vm.groupby("p"):
        fr = FMETA[FMETA.p == p]
        if len(g) < 2 or not len(fr):
            continue
        vs = rng.permutation(g.index.values)[:m]
        for j, v in enumerate(vs):
            other = fr[fr.video != VMETA.video[v]]
            if not len(other):
                continue
            f = rng.choice(other.index.values)
            if j < len(vs) // 2:
                Fi.append(f); Vi.append(v); Yl.append(1)
            else:
                nf.append(f); nv.append(v)
    nv = list(rng.permutation(nv))
    for f in rng.permutation(nf):
        pf = FMETA.p[f]
        ok = [j for j, v in enumerate(nv) if VMETA.p[v] != pf and (not sg or GEN.get(VMETA.p[v]) == GEN.get(pf))]
        if ok:
            Fi.append(f); Vi.append(nv.pop(ok[0])); Yl.append(0)
    Fi, Vi = np.array(Fi, int), np.array(Vi, int)
    d = dict(f={k: FA[k][Fi] for k in ("vgg", "ibv", "agef", "arcface")}, v={k: VA[k][Vi] for k in ("btc", "iba", "agev", "ecapa192")},
             y=np.array(Yl), pf=FMETA.p.values[Fi], pv=VMETA.p.values[Vi])
    if len(d["y"]) >= 20 and 0 < d["y"].mean() < 1:
        d["cf"] = clus(cent(d["f"]["arcface"]), TF_CLUS); d["cv"] = clus(cent(d["v"]["ecapa192"]), TV_CLUS)
        return d
    return None


def score(model, d):
    Af, Av = d["f"]["vgg"], d["v"]["btc"]
    embs = [embed(net, prep, Af, Av, prep.f(model["TF"]).mean(0), prep.v(model["TV"]).mean(0)) for net, prep in model["nets"]]
    M = np.mean([zm(a @ b.T) for a, b in embs], 0)
    cx = model["cca"].transform_x(Af - Af.mean(0) + model["mf"]); cy = model["cca"].transform_y(Av - Av.mean(0) + model["mv"])
    M = (1 - W_CCA) * M + W_CCA * zm(cx @ cy.T)
    Fb, Ab = d["f"]["ibv"], d["v"]["iba"]
    ib = l2n(Fb - Fb.mean(0)) @ l2n(Ab - Ab.mean(0)).T
    S = zm((1 - W_IB) * zm(M) + W_IB * zm(ib)) + W_AGE * zm(-np.abs(d["f"]["agef"][:, None] - d["v"]["agev"][None, :]))
    cfo, cvo = pd.factorize(d["pf"])[0], pd.factorize(d["pv"])[0]
    out = {}
    for var, X in (("member", M), ("prod", S)):
        out[(var, "sample")] = eer(np.diag(X), d["y"])
        out[(var, "person")] = eer(block(X, cfo, cvo), d["y"])
        out[(var, "cluster")] = eer(block(X, d["cf"], d["cv"]), d["y"])
    return out


RES = OUT / "cur_results.csv"
prev = [p for p in IN.rglob("cur_results.csv") if OUT not in p.parents]
if prev and not RES.exists():
    shutil.copy(prev[0], RES); log("resume from", prev[0])
recs = pd.read_csv(RES).to_dict("records") if RES.exists() else []
done = {(r["fold"], r["arm"]) for r in recs}
for s in FOLDS:
    held = [p for p in PERSONS if PFOLD[p] == s - 1]
    pool = [p for p in PERSONS if PFOLD[p] != s - 1]
    rng_e = np.random.RandomState(s)
    files = {}
    for lang in ("urdu", "hindi", "english"):
        for sg in (False, True):
            if lang == "hindi" and sg:
                continue
            d = eval_file(held, lang, sg, rng_e)
            if d is not None:
                files[(lang, "g" if sg else "ng")] = d
    like = v4_likeness(pool)
    half = len(like) // 2
    matched = list(like.sort_values(ascending=False).index[:half])
    randomh = list(np.random.RandomState(500 + s).permutation(sorted(like.index))[:half])
    log(f"fold {s}: held {len(held)}, pool {len(pool)}, files {list(files)}; XEN_MATCH / XEN_RAND {half} persons, overlap {len(set(matched) & set(randomh))}")
    for arm in ARMS:
        if (s, arm) in done:
            continue
        t0 = time.time(); rng = np.random.RandomState(4000 + s)
        TF, TV, TS = V4["vgg"], V4["btc"], V4S
        ex = {"R": None, "XALL": (pool, None, None), "XEN": (pool, ["english"], None), "XEN_MATCH": (matched, ["english"], None),
              "XEN_RAND": (randomh, ["english"], None), "XLANG_EQ": (pool, None, CAP_LANG)}[arm]
        if ex is not None:
            EF, EV, ES = ext_rows(ex[0], langs=ex[1], per_lang=ex[2], rng=rng)
            TF, TV, TS = np.concatenate([TF, EF]), np.concatenate([TV, EV]), np.concatenate([TS, ES])
        bpe = max(1, len(TS) // (P_PK * K_PK)); ep = max(1, int(round(STEPS / bpe)))
        nets = [train_one(TF, TV, TS, seed=s + 100 * i, epochs=ep) for i in range(N_NETS)]
        model = dict(nets=nets, cca=RidgeCCA().fit(TF, TV), mf=TF.mean(0), mv=TV.mean(0), TF=TF, TV=TV)
        for (lang, prot), d in files.items():
            for (var, level), e in score(model, d).items():
                recs.append(dict(fold=s, arm=arm, train_ids=len(set(TS)), train_rows=len(TS), steps=bpe * ep, data=lang,
                                 protocol=prot, variant=var, level=level, n=len(d["y"]), eer=e))
        pd.DataFrame(recs).to_csv(RES, index=False)
        log(f"fold {s} {arm:10s} ids {len(set(TS)):4d} rows {len(TS):6d} steps {bpe * ep:5d}  {time.time() - t0:.0f}s")

## 6. Paired summary

In [ ]:
# ============================================================================ paired summary (+ = candidate better)
R_ = pd.read_csv(RES)
W = R_.pivot_table(index=["variant", "level", "data", "protocol", "fold"], columns="arm", values="eer")
rows = []
for (a, b) in [("XALL", "R"), ("XEN", "R"), ("XEN_MATCH", "R"), ("XEN_RAND", "R"), ("XLANG_EQ", "R"),
               ("XEN_MATCH", "XEN_RAND"), ("XEN", "XALL"), ("XLANG_EQ", "XALL")]:
    if a not in W or b not in W:
        continue
    d = (W[b] - W[a]).groupby(level=[0, 1, 2, 3])
    for k, g in d:
        rows.append(dict(comparison=f"{a} vs {b}", variant=k[0], level=k[1], data=k[2], protocol=k[3],
                         gain=round(g.mean(), 2), wins=int((g > 0).sum()), folds=int(g.notna().sum())))
S_ = pd.DataFrame(rows); S_.to_csv(OUT / "cur_summary.csv", index=False)
show = S_[(S_.variant == "prod") & S_.level.isin(["person", "cluster"])]
print(show.assign(cell=show.data + " " + show.protocol + " " + show.level).pivot_table(
    index="cell", columns="comparison", values="gain").round(2).to_string())
print("\nwins / folds (prod):")
print(show.assign(cell=show.data + " " + show.protocol + " " + show.level, wf=show.wins.astype(str) + "/" + show.folds.astype(str)).pivot_table(
    index="cell", columns="comparison", values="wf", aggfunc="first").to_string())
print("\nNEXT: download /kaggle/working/eda_cur (Output tab) and hand it back: eda_ext_persons.csv, eda_ext_summary.json, shift_c2st.csv, cur_results.csv, cur_summary.csv")